In [2]:
import os
import json
import csv

# --- CONFIGURATION ---
# Read your chunks from the imported dataset folder
INPUT_DIR = "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-7b-naive-all-results"

# Write the final merged files and CSV to the Kaggle working directory
OUTPUT_DIR = "/kaggle/working"

# The 4 quadrants of your ablation study
RUN_CONFIGS = {
    "3B_EXTREME_28x28": [
        "3B_ablation_person_A_EXTREME.json", "3B_ablation_person_B_EXTREME.json", 
        "3B_ablation_person_C_EXTREME.json", "3B_ablation_person_D_EXTREME.json"
    ],
    "7B_EXTREME_28x28": [
        "7B_ablation_person_A_EXTREME.json", "7B_ablation_person_B_EXTREME.json", 
        "7B_ablation_person_C_EXTREME.json", "7B_ablation_person_D_EXTREME.json"
    ],
    "3B_STANDARD_224x224": [
        "3B_ablation_person_A_STANDARD.json", "3B_ablation_person_B_STANDARD.json", 
        "3B_ablation_person_C_STANDARD.json", "3B_ablation_person_D_STANDARD.json"
    ],
    "7B_STANDARD_224x224": [
        "7B_ablation_person_A_STANDARD.json", "7B_ablation_person_B_STANDARD.json", 
        "7B_ablation_person_C_STANDARD.json", "7B_ablation_person_D_STANDARD.json"
    ]
}

MASTER_CSV_OUTPUT = os.path.join(OUTPUT_DIR, "MASTER_ABLATION_RESULTS.csv")

# MVBench Task Mapping by Index
MVBENCH_TASKS = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

def get_task_name(index):
    task_idx = index // 200
    if 0 <= task_idx < len(MVBENCH_TASKS):
        return MVBENCH_TASKS[task_idx]
    return "unknown_task"

print("Commencing Master Grand Aggregation...\n")

# Data structure to hold all aggregated metrics
# format: final_metrics[task_name][run_name] = {"acc": X, "tps": Y, "ttft": Z}
final_metrics = {task: {run: {"correct": 0, "total": 0, "tps_sum": 0.0, "ttft_sum": 0.0} for run in RUN_CONFIGS} for task in MVBENCH_TASKS}
overall_metrics = {run: {"correct": 0, "total": 0, "tps_sum": 0.0, "ttft_sum": 0.0, "peak_vram": 0.0} for run in RUN_CONFIGS}

# --- PHASE 1: MERGE & CALCULATE ---
for run_name, file_list in RUN_CONFIGS.items():
    print(f"Processing Quadrant: {run_name}")
    master_records = {} # Use Dictionary Upsert to ensure uniqueness
    
    # 1. Load and Merge A, B, C, D
    for file_name in file_list:
        file_path = os.path.join(INPUT_DIR, file_name)  # <-- READING FROM INPUT_DIR
        if os.path.exists(file_path):
            with open(file_path, "r") as f:
                try:
                    data = json.load(f)
                    for record in data:
                        master_records[record["id"]] = record
                except json.JSONDecodeError:
                    print(f"  [!] ERROR: {file_name} is corrupted.")
        else:
            print(f"  [-] WARNING: {file_name} not found.")
            
    # Sort chronologically
    sorted_records = [master_records[k] for k in sorted(master_records.keys())]
    
    # Save the consolidated JSON for this specific run
    merged_json_path = os.path.join(OUTPUT_DIR, f"MASTER_{run_name}.json") # <-- WRITING TO OUTPUT_DIR
    with open(merged_json_path, "w") as f:
        json.dump(sorted_records, f, indent=4)
    print(f"  [+] Merged {len(sorted_records)} records into {merged_json_path}")
    
    # 2. Calculate Metrics
    for r in sorted_records:
        task_name = get_task_name(r["id"])
        is_correct = (r["response"].strip().upper() == r.get("ground_truth_letter", "").strip().upper())
        
        # Add to Task Metrics
        if is_correct:
            final_metrics[task_name][run_name]["correct"] += 1
            overall_metrics[run_name]["correct"] += 1
            
        final_metrics[task_name][run_name]["total"] += 1
        overall_metrics[run_name]["total"] += 1
        
        final_metrics[task_name][run_name]["tps_sum"] += r.get("tokens_per_second", 0.0)
        overall_metrics[run_name]["tps_sum"] += r.get("tokens_per_second", 0.0)
        
        final_metrics[task_name][run_name]["ttft_sum"] += r.get("ttft_seconds", 0.0)
        overall_metrics[run_name]["ttft_sum"] += r.get("ttft_seconds", 0.0)
        
        if r.get("peak_vram_gb", 0.0) > overall_metrics[run_name]["peak_vram"]:
            overall_metrics[run_name]["peak_vram"] = r.get("peak_vram_gb", 0.0)

# --- PHASE 2: GENERATE COMPARATIVE CSV ---
print("\nGenerating Master CSV...")

csv_headers = [
    "Task", 
    "3B_28x28_Acc(%)", "7B_28x28_Acc(%)", "3B_224x224_Acc(%)", "7B_224x224_Acc(%)",
    "3B_28x28_TPS", "7B_28x28_TPS", "3B_224x224_TPS", "7B_224x224_TPS"
]

csv_data = []

for task in MVBENCH_TASKS:
    row = [task]
    # Format Accuracy
    for run in RUN_CONFIGS.keys():
        data = final_metrics[task][run]
        acc = round((data["correct"] / data["total"] * 100), 2) if data["total"] > 0 else 0.0
        row.append(acc)
    # Format TPS
    for run in RUN_CONFIGS.keys():
        data = final_metrics[task][run]
        tps = round((data["tps_sum"] / data["total"]), 2) if data["total"] > 0 else 0.0
        row.append(tps)
    csv_data.append(row)

# Append Overall Row
overall_row = ["OVERALL"]
for run in RUN_CONFIGS.keys():
    data = overall_metrics[run]
    acc = round((data["correct"] / data["total"] * 100), 2) if data["total"] > 0 else 0.0
    overall_row.append(acc)
for run in RUN_CONFIGS.keys():
    data = overall_metrics[run]
    tps = round((data["tps_sum"] / data["total"]), 2) if data["total"] > 0 else 0.0
    overall_row.append(tps)

# Append Peak VRAM Row
vram_row = ["PEAK_VRAM_GB"]
for run in RUN_CONFIGS.keys():
    vram_row.append(overall_metrics[run]["peak_vram"])
for run in RUN_CONFIGS.keys():
    vram_row.append("-") # TPS columns blank for VRAM row

with open(MASTER_CSV_OUTPUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(csv_headers)
    writer.writerows(csv_data)
    writer.writerow([]) # blank line
    writer.writerow(overall_row)
    writer.writerow(vram_row)

print(f"Aggregation Complete! The holy grail dataset is saved at:")
print(f"--> {MASTER_CSV_OUTPUT}")

Commencing Master Grand Aggregation...

Processing Quadrant: 3B_EXTREME_28x28
  [+] Merged 3800 records into /kaggle/working/MASTER_3B_EXTREME_28x28.json
Processing Quadrant: 7B_EXTREME_28x28
  [+] Merged 3800 records into /kaggle/working/MASTER_7B_EXTREME_28x28.json
Processing Quadrant: 3B_STANDARD_224x224
  [+] Merged 3800 records into /kaggle/working/MASTER_3B_STANDARD_224x224.json
Processing Quadrant: 7B_STANDARD_224x224
  [+] Merged 3800 records into /kaggle/working/MASTER_7B_STANDARD_224x224.json

Generating Master CSV...
Aggregation Complete! The holy grail dataset is saved at:
--> /kaggle/working/MASTER_ABLATION_RESULTS.csv
